In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

In [2]:
DATA_PATH = Path("../data/raw/WineQT.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [3]:
TARGET = "quality"
ID_COL = "Id"

y = df[TARGET]
X = df.drop(columns=[TARGET, ID_COL])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [5]:
target_summary = pd.DataFrame({
    "train": y_train.describe(),
    "test": y_test.describe(),
})

target_summary

,train,test
count,914.000000,229.000000
mean,5.657549,5.655022
std,0.806472,0.804992
min,3.000000,3.000000
25%,5.000000,5.000000
50%,6.000000,6.000000
75%,6.000000,6.000000
max,8.000000,8.000000


In [6]:
quality_distribution_split = pd.concat(
    [
        y_train.value_counts(normalize=True).sort_index().rename("train_proportion"),
        y_test.value_counts(normalize=True).sort_index().rename("test_proportion"),
    ],
    axis=1,
)

quality_distribution_split

,train_proportion,test_proportion
quality,,
3,0.005470,0.004367
4,0.028446,0.030568
5,0.422319,0.423581
6,0.404814,0.401747
7,0.124726,0.126638
8,0.014223,0.013100


The holdout test set was created using the same split logic as Stage 3.
It remains untouched and will not be used for model evaluation in this stage.

Шаг 4.5 — CV and metrics

In [7]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

Шаг 4.6 — define models

In [8]:
models = {
    "linear_regression": LinearRegression(),
    
    "ridge_scaled": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    
    "decision_tree_depth_3": DecisionTreeRegressor(
        max_depth=3,
        random_state=42,
    ),
    
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    
    "extra_trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    
    "gradient_boosting": GradientBoostingRegressor(
        random_state=42,
    ),
    
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        random_state=42,
    ),
    
    "svr_scaled": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", SVR()),
    ]),
}

Шаг 4.7 — evaluation function

In [9]:
def evaluate_model_cv(model, X_train, y_train, cv, scoring):
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )
    
    result = {
        "mae_mean": -cv_results["test_mae"].mean(),
        "mae_std": cv_results["test_mae"].std(),
        "rmse_mean": -cv_results["test_rmse"].mean(),
        "rmse_std": cv_results["test_rmse"].std(),
        "r2_mean": cv_results["test_r2"].mean(),
        "r2_std": cv_results["test_r2"].std(),
    }
    
    return result

Шаг 4.8 — run model comparison

In [10]:
results = []

for model_name, model in models.items():
    model_result = evaluate_model_cv(
        model=model,
        X_train=X_train,
        y_train=y_train,
        cv=cv,
        scoring=scoring,
    )
    
    model_result["model"] = model_name
    results.append(model_result)

comparison_results = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values("rmse_mean")
)

comparison_results

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
extra_trees,0.425218,0.025101,0.588285,0.047960,0.463178,0.028018
random_forest,0.451219,0.035861,0.596684,0.057495,0.448979,0.034305
gradient_boosting,0.482916,0.032341,0.621754,0.059080,0.401770,0.034126
hist_gradient_boosting,0.467354,0.028752,0.621969,0.050674,0.400221,0.026282
svr_scaled,0.469236,0.018740,0.631291,0.035359,0.379569,0.035737
ridge_scaled,0.507413,0.020749,0.650699,0.047391,0.343040,0.018915
linear_regression,0.507391,0.020684,0.650763,0.047325,0.342897,0.018980
decision_tree_depth_3,0.557753,0.033532,0.704902,0.063687,0.230518,0.041134


In [11]:
comparison_results.round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
extra_trees,0.4252,0.0251,0.5883,0.0480,0.4632,0.0280
random_forest,0.4512,0.0359,0.5967,0.0575,0.4490,0.0343
gradient_boosting,0.4829,0.0323,0.6218,0.0591,0.4018,0.0341
hist_gradient_boosting,0.4674,0.0288,0.6220,0.0507,0.4002,0.0263
svr_scaled,0.4692,0.0187,0.6313,0.0354,0.3796,0.0357
ridge_scaled,0.5074,0.0207,0.6507,0.0474,0.3430,0.0189
linear_regression,0.5074,0.0207,0.6508,0.0473,0.3429,0.0190
decision_tree_depth_3,0.5578,0.0335,0.7049,0.0637,0.2305,0.0411


Шаг 4.9 — compare against Ridge baseline

In [12]:
ridge_rmse = comparison_results.loc["ridge_scaled", "rmse_mean"]
ridge_mae = comparison_results.loc["ridge_scaled", "mae_mean"]
ridge_r2 = comparison_results.loc["ridge_scaled", "r2_mean"]

comparison_vs_ridge = comparison_results.copy()

comparison_vs_ridge["rmse_delta_vs_ridge"] = (
    comparison_vs_ridge["rmse_mean"] - ridge_rmse
)

comparison_vs_ridge["mae_delta_vs_ridge"] = (
    comparison_vs_ridge["mae_mean"] - ridge_mae
)

comparison_vs_ridge["r2_delta_vs_ridge"] = (
    comparison_vs_ridge["r2_mean"] - ridge_r2
)

comparison_vs_ridge.round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,rmse_delta_vs_ridge,mae_delta_vs_ridge,r2_delta_vs_ridge
model,,,,,,,,,
extra_trees,0.4252,0.0251,0.5883,0.0480,0.4632,0.0280,-0.0624,-0.0822,0.1201
random_forest,0.4512,0.0359,0.5967,0.0575,0.4490,0.0343,-0.0540,-0.0562,0.1059
gradient_boosting,0.4829,0.0323,0.6218,0.0591,0.4018,0.0341,-0.0289,-0.0245,0.0587
hist_gradient_boosting,0.4674,0.0288,0.6220,0.0507,0.4002,0.0263,-0.0287,-0.0401,0.0572
svr_scaled,0.4692,0.0187,0.6313,0.0354,0.3796,0.0357,-0.0194,-0.0382,0.0365
ridge_scaled,0.5074,0.0207,0.6507,0.0474,0.3430,0.0189,0.0000,0.0000,0.0000
linear_regression,0.5074,0.0207,0.6508,0.0473,0.3429,0.0190,0.0001,-0.0000,-0.0001
decision_tree_depth_3,0.5578,0.0335,0.7049,0.0637,0.2305,0.0411,0.0542,0.0503,-0.1125


Шаг 4.10 — select top candidates for later tuning

In [13]:
top_candidates_by_rmse = comparison_results.head(3)

top_candidates_by_rmse.round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
extra_trees,0.4252,0.0251,0.5883,0.0480,0.4632,0.0280
random_forest,0.4512,0.0359,0.5967,0.0575,0.4490,0.0343
gradient_boosting,0.4829,0.0323,0.6218,0.0591,0.4018,0.0341


## Controlled model comparison conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains all physicochemical columns except `quality` and `Id`.
- The same train/test split as Stage 3 was used:
  - `test_size=0.2`;
  - `random_state=42`;
  - `stratify=y`.
- The holdout test set was not evaluated.
- Model comparison was performed using 5-fold cross-validation on `X_train` only.

### Models compared

Baseline models:

- `LinearRegression`;
- `Ridge(alpha=1.0)` with `StandardScaler` inside Pipeline;
- `DecisionTreeRegressor(max_depth=3)`.

Stronger candidates:

- `RandomForestRegressor`;
- `ExtraTreesRegressor`;
- `GradientBoostingRegressor`;
- `HistGradientBoostingRegressor`;
- `SVR` with `StandardScaler` inside Pipeline.

### Metrics

- MAE;
- RMSE;
- R².

### Main findings

Write here:

- best model by RMSE;
- best model by MAE;
- whether nonlinear models improved over Ridge;
- whether tree ensembles improved over the shallow decision tree;
- whether SVR was competitive;
- which 2–3 candidates should be carried forward to tuning.

### Leakage control

- `Id` was excluded from the feature matrix.
- The holdout test set was not evaluated.
- Cross-validation was performed only on `X_train`.
- Scaling for Ridge and SVR was inside Pipeline.
- No preprocessing was fitted on the full dataset.
- No outliers were removed.
- No target transformation was applied.
- Predictions were not rounded for metric calculation.
- No hyperparameter tuning was performed.
- Multiclass and binary framings were not used.

## Controlled model comparison conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains all physicochemical columns except `quality` and `Id`.
- The same train/test split logic as Stage 3 was used:
  - `test_size=0.2`;
  - `random_state=42`;
  - `stratify=y`.
- The holdout test set was not evaluated.
- Model comparison was performed using 5-fold cross-validation on `X_train` only.

### Models compared

Baseline models:

- `LinearRegression`;
- `Ridge(alpha=1.0)` with `StandardScaler` inside Pipeline;
- `DecisionTreeRegressor(max_depth=3)`.

Stronger candidates:

- `RandomForestRegressor`;
- `ExtraTreesRegressor`;
- `GradientBoostingRegressor`;
- `HistGradientBoostingRegressor`;
- `SVR` with `StandardScaler` inside Pipeline.

### Metrics

- MAE;
- RMSE;
- R².

### Main findings

- The best model by RMSE was `ExtraTreesRegressor`:
  - MAE ≈ 0.4252;
  - RMSE ≈ 0.5883;
  - R² ≈ 0.4632.
- The best model by MAE was also `ExtraTreesRegressor`.
- `RandomForestRegressor` was the second-best model:
  - MAE ≈ 0.4512;
  - RMSE ≈ 0.5967;
  - R² ≈ 0.4490.
- Both Extra Trees and Random Forest clearly improved over the Ridge baseline.
- Compared with `ridge_scaled`, `extra_trees` improved:
  - RMSE by approximately 0.0624;
  - MAE by approximately 0.0822;
  - R² by approximately 0.1201.
- Tree ensembles substantially outperformed the shallow `DecisionTreeRegressor(max_depth=3)`, which suggests that nonlinear patterns are useful but a single shallow tree is too limited.
- Boosting models also improved over Ridge, but less strongly than Extra Trees and Random Forest in this controlled comparison.
- `SVR` with scaling improved over Ridge but was not competitive with the best tree ensembles.
- Nonlinear model families help on this dataset.
- The most promising candidates for later tuning are:
  - `ExtraTreesRegressor`;
  - `RandomForestRegressor`;
  - `HistGradientBoostingRegressor` or `GradientBoostingRegressor`.

### Candidate selection for later stages

Recommended candidates to carry forward:

1. `ExtraTreesRegressor`
   - Best MAE and RMSE in Stage 4.
   - Strongest improvement over Ridge.

2. `RandomForestRegressor`
   - Second-best RMSE and strong R².
   - Useful robust ensemble baseline.

3. `HistGradientBoostingRegressor` / `GradientBoostingRegressor`
   - Boosting family improved over Ridge.
   - Worth testing during tuning because boosting models may benefit more from hyperparameter search.

### Leakage control

- `Id` was excluded from the feature matrix.
- The holdout test set was not evaluated.
- Cross-validation was performed only on `X_train`.
- Scaling for Ridge and SVR was inside Pipeline.
- No preprocessing was fitted on the full dataset.
- No outliers were removed.
- No target transformation was applied.
- Predictions were not rounded for metric calculation.
- No hyperparameter tuning was performed.
- Multiclass and binary framings were not used.